# Baseline Knowledge Graph Recommender System (MovieLens 100k)
This notebook implements a baseline knowledge graph-based recommendation system using the MovieLens 100k dataset. It uses TransE and DistMult knowledge graph embedding models via PyKEEN to generate and evaluate link predictions. This serves as the foundational implementation before introducing memory decay and forgetting mechanisms.

In [ ]:
!pip install gliner

In [1]:
import pandas as pd
from tqdm import tqdm


merge_entities, is designed to merge consecutive or adjacent entities that share the same label and are part of the same logical entity (such as a movie title or name) in Named Entity Recognition (NER) output.

In [4]:
from gliner import GLiNER

def merge_entities(entities):
    if not entities:
        return []

    merged = []
    current = entities[0]

    for next_entity in entities[1:]:
        if next_entity['label'] == current['label'] and (
            next_entity['start'] == current['end'] or
            next_entity['start'] == current['end'] + 1
        ):
            current['text'] = text[current['start']: next_entity['end']].strip()
            current['end'] = next_entity['end']
        else:
            merged.append(current)
            current = next_entity

    merged.append(current)
    return merged


In [5]:
# Load the MovieLens 100K dataset files
u_data = pd.read_csv("u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
u_item = pd.read_csv("u.item", sep="|", names=["item_id", "movie_title", "release_date", "video_release_date",
                                               "IMDb_url", "unknown", "Action", "Adventure", "Animation", 
                                               "Children's", "Comedy", "Crime", "Documentary", "Drama", 
                                               "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery", 
                                               "Romance", "Sci-Fi", "Thriller", "War", "Western"], encoding="latin-1")

# Merge user ratings with movie titles and genres
merged_data = pd.merge(u_data, u_item[["item_id", "movie_title"]], on="item_id")

# Extract genres from u.item (the columns starting from 6th column onward)
genres = u_item.columns[6:]  # All columns related to genres

# Merge the genres data with the merged dataset
merged_data = pd.merge(merged_data, u_item[["item_id"] + list(genres)], on="item_id")

# Check the first few rows of merged data
print(merged_data.head())

# Extracting movie titles with genres
for _, row in merged_data.iterrows():
    movie_title = row["movie_title"]
    movie_genres = [genre for genre in genres if row[genre] == 1]
    print(f"Movie: {movie_title} - Genres: {', '.join(movie_genres)}")

# Example: Filter movies based on a genre (e.g., Comedy)
comedy_movies = merged_data[merged_data["Comedy"] == 1]
print("\nComedy Movies:")
for _, row in comedy_movies.iterrows():
    print(row["movie_title"])

# Example: Filter movies with ratings 4 or higher
high_rated_movies = merged_data[merged_data["rating"] >= 4]
print("\nHigh-Rated Movies (rating >= 4):")
for _, row in high_rated_movies.iterrows():
    movie_title = row["movie_title"]
    rating = row["rating"]
    print(f"{movie_title} - Rating: {rating}")

   user_id  item_id  rating  timestamp                 movie_title  Action  \
0      196      242       3  881250949                Kolya (1996)       0   
1      186      302       3  891717742    L.A. Confidential (1997)       0   
2       22      377       1  878887116         Heavyweights (1994)       0   
3      244       51       2  880606923  Legends of the Fall (1994)       0   
4      166      346       1  886397596         Jackie Brown (1997)       0   

   Adventure  Animation  Children's  Comedy  ...  Fantasy  Film-Noir  Horror  \
0          0          0           0       1  ...        0          0       0   
1          0          0           0       0  ...        0          1       0   
2          0          0           1       1  ...        0          0       0   
3          0          0           0       0  ...        0          0       0   
4          0          0           0       0  ...        0          0       0   

   Musical  Mystery  Romance  Sci-Fi  Thriller  Wa

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Movie: Natural Born Killers (1994) - Genres: Action, Thriller
Movie: Angels in the Outfield (1994) - Genres: Children's, Comedy
Movie: Safe (1995) - Genres: Thriller
Movie: Lawrence of Arabia (1962) - Genres: Adventure, War
Movie: Clear and Present Danger (1994) - Genres: Action, Adventure, Thriller
Movie: Hudsucker Proxy, The (1994) - Genres: Comedy, Romance
Movie: Jane Eyre (1996) - Genres: Drama, Romance
Movie: Titanic (1997) - Genres: Action, Drama, Romance
Movie: Jungle Book, The (1994) - Genres: Adventure, Children's, Romance
Movie: People vs. Larry Flynt, The (1996) - Genres: Drama
Movie: Sound of Music, The (1965) - Genres: Musical
Movie: Braveheart (1995) - Genres: Action, Drama, War
Movie: Abyss, The (1989) - Genres: Action, Adventure, Sci-Fi, Thriller
Movie: Pretty Woman (1990) - Genres: Comedy, Romance
Movie: Star Trek: The Wrath of Khan (1982) - Genres: Action, Adventure, Sci-Fi
Movie: While You Were Sleeping (1995) - Genres: Comedy, Romance
Movie: Emma (1996) - Genres: Dr

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Carlito's Way (1993) - Rating: 4
English Patient, The (1996) - Rating: 4
On Golden Pond (1981) - Rating: 4
Dirty Dancing (1987) - Rating: 5
Forrest Gump (1994) - Rating: 5
Ice Storm, The (1997) - Rating: 5
To Kill a Mockingbird (1962) - Rating: 5
Clear and Present Danger (1994) - Rating: 5
Postino, Il (1994) - Rating: 5
Flirting With Disaster (1996) - Rating: 4
Independence Day (ID4) (1996) - Rating: 5
Dead Poets Society (1989) - Rating: 4
Hoodlum (1997) - Rating: 4
Star Wars (1977) - Rating: 5
Babe (1995) - Rating: 5
Shine (1996) - Rating: 4
Back to the Future (1985) - Rating: 5
This Is Spinal Tap (1984) - Rating: 4
My Life as a Dog (Mitt liv som hund) (1985) - Rating: 4
Schindler's List (1993) - Rating: 4
Star Trek: First Contact (1996) - Rating: 4
Desperado (1995) - Rating: 5
Alien (1979) - Rating: 4
Contact (1997) - Rating: 4
Dead Man Walking (1995) - Rating: 4
Nightmare on Elm Street, A (1984) - Rating: 4
Eye for an Eye (1996) - Rating: 5
Twelve Monkeys (1995) - Rating: 4
Leading 

In [ ]:
!pip install pyvis

In [ ]:
from pyvis.network import Network
import networkx as nx

# Create a graph
G = nx.Graph()

# Helper functions for node attributes
def get_color(node):
    return "blue" if node.startswith("user_") else "green"

def get_size(node):
    return 10 if node.startswith("user_") else 20

# Create triples and add them to the graph
for _, row in merged_data.iterrows():
    user_node = f"user_{row['user_id']}"
    movie_node = row['movie_title']
    rating = row['rating']
    
    # Add nodes for user and movie
    G.add_node(user_node, title=user_node, color=get_color(user_node), size=get_size(user_node), label=user_node)
    G.add_node(movie_node, title=movie_node, color=get_color(movie_node), size=get_size(movie_node), label=movie_node)
    
    # Add edge (rating relationship between user and movie)
    G.add_edge(user_node, movie_node, title=f"rated {rating}", weight=rating)

# Create the pyvis network visualization
nt = Network(height="750px", width="100%", bgcolor="#222222", font_color="white")
nt.from_nx(G)
nt.force_atlas_2based(central_gravity=0.015, gravity=-31)

# Show the network visualization in an HTML file
nt.show("graph.html", notebook=False)

# Display the graph in a Jupyter Notebook (optional)
from IPython.display import IFrame
IFrame("graph.html", width=1000, height=800)

## Conclusion
